# CaptionBench: the grounding-gap study in one notebook

This notebook runs the whole CSE434 project on a Colab GPU.
It needs no account, no API key, and no uploaded file. Google Drive is optional.

Choose **Runtime > Run all**. You do not need to change a setting.

The notebook asks Colab for a T4 GPU. If the GPU check cell reports no GPU, choose **Runtime > Change runtime type > T4 GPU**, then choose **Runtime > Run all** again.

The default profile is `quick`, a short demo. It ends with a table of scores and a page of photos with each model's caption under each photo.
The quick scores are a wiring check. They are not study results. For the study, set `PROFILE` to `midterm` or `full` in the Settings cell.

The notebook copies the public code, downloads Flickr8k, trains and scores every model, and shows the tables, the figures, and a page of real captions.
The last cells download a zip file of the results and release the GPU.

| Profile | What runs | Approximate time on a T4 |
|---|---|---|
| `quick` | One small decoder, zero-shot BLIP base, a short fine-tune. A wiring check. Its scores are not study results. | 15 to 20 minutes |
| `midterm` | The conditions in the midterm report: 6 decoders, BLIP base and large, 3 fine-tuned BLIP base seeds. | 3 to 4 hours |
| `full` | `midterm`, plus a wider decoder on ConvNeXt features and a fine-tuned BLIP large. | 5 to 6 hours |
| `improve` | Our decoders against released BLIP: a wider decoder, three stronger frozen image encoders (MobileNetV3, ConvNeXt, DINOv2), and beam search. 15 short training runs. No fine-tuning. | 30 to 60 minutes |

The times are estimates from a local RTX 5070 run. A Colab run has not timed them. The `improve` time is an estimate that no run has measured.
Fine-tuning takes most of the time. Set `FINETUNE_SEEDS` to `17` to run one seed in about a third of the time.

**A long run checks itself first.** The `midterm` and `full` profiles run every later stage once on a tiny fine-tune before the long fine-tuning starts.
A wiring fault then stops the run in the first minutes, not in the last.

**Compute units.** With `AUTO_DISCONNECT` on, the last cell releases the GPU when the run ends or fails.

**Google Drive (optional).** With `SAVE_TO_DRIVE` off, every file stays in the runtime and a new runtime starts from the beginning.
With `SAVE_TO_DRIVE` on, the notebook asks for Drive access once. It then copies the small result files to `My Drive/CaptionBench` after each stage.
A later run continues from those files. The copy holds captions, scores, figures, and the caption gallery. It holds no model weights.

In [ ]:
# @title Settings
# "quick" is the short demo. "midterm" (hours) and "full" (longer) are the long study runs.
# "improve" trains stronger versions of our own decoder and compares them with released BLIP.
PROFILE = "quick"  # @param ["quick", "midterm", "full", "improve"]
FINETUNE_SEEDS = "17 29 43"  # @param {type:"string"}
SAVE_TO_DRIVE = False  # @param {type:"boolean"}
AUTO_DISCONNECT = True  # @param {type:"boolean"}
REPO_REF = "midterm-checkin"  # @param {type:"string"}
REPO_URL = "https://github.com/jhunter11/cse434-fp.git"

In [ ]:
# @title Check the GPU
import torch

if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    print(f"GPU: {properties.name}, {properties.total_memory / 2**30:.1f} GiB. torch {torch.__version__}")
    if PROFILE == "full" and properties.total_memory < 15 * 2**30:
        print("The full profile fine-tunes BLIP large. Use a GPU with 16 GiB or more, or choose midterm.")
else:
    print("No GPU is connected. Choose Runtime > Change runtime type > T4 GPU, then choose Runtime > Run all again.")
    raise RuntimeError("No GPU is connected. The cells below stop here.")

In [ ]:
# @title Get the code
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
DRIVE_RUN = None
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_RUN = Path("/content/drive/MyDrive/CaptionBench") / f"study-{PROFILE}"
    DRIVE_RUN.mkdir(parents=True, exist_ok=True)
if IN_COLAB:
    REPO = Path("/content/cse434-fp")
    if not REPO.is_dir():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO)], check=True)
    else:
        # A runtime that already has the code takes the newest commit of the branch.
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "--depth", "1", "origin", REPO_REF], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[models]", "matplotlib", "pandas"],
                   check=True)
else:
    REPO = Path.cwd() if (Path.cwd() / "scripts" / "run_study.py").is_file() else Path.cwd().parent
os.chdir(REPO)
os.environ["PYTHONPATH"] = str(REPO / "src")
WORK = REPO / "outputs" / f"study-{PROFILE}"
WORK.mkdir(parents=True, exist_ok=True)
SMALL_FILES = ("*.jsonl", "*.done", "*.json", "*.log", "gallery.*", "runs/*/*.json", "results/*", "report/*",
               "report/figures/*", "preflight/passed", "preflight/results/*")


def sync(source, target):
    """Copy the small result files. Model weights and feature caches stay in the runtime."""
    copied = 0
    for pattern in SMALL_FILES:
        for path in source.glob(pattern):
            destination = target / path.relative_to(source)
            if path.is_file() and (not destination.exists() or destination.stat().st_size != path.stat().st_size
                                   or destination.stat().st_mtime < path.stat().st_mtime):
                destination.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, destination)
                copied += 1
    return copied


if DRIVE_RUN:
    print("Restored", sync(DRIVE_RUN, WORK), "files from", DRIVE_RUN)
print("Code:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

## Run the study

This cell does all the work. It prints one `===` line when a stage starts and one when the stage ends.
The first stage downloads Flickr8k (1.1 GiB) from a public mirror and checks the SHA256 of each archive.
The 1,000 test images stay sealed. No stage reads them.
The full output goes to `run.log` in the results zip.

In [ ]:
# @title Download the data, train, and score
SHOWN = ("===", "epoch", "{", "Study", "Tables", "Aggregate", "Gallery", "The quick", "  Flickr", "Downloading",
         "Extracted", "Preflight")
command = [sys.executable, "-u", "scripts/run_study.py", "--profile", PROFILE, "--work", str(WORK),
           "--finetune-seeds", *FINETUNE_SEEDS.split()]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=REPO)
tail = []
with open(WORK / "run.log", "a", encoding="utf-8") as log:
    for line in process.stdout:
        log.write(line)
        log.flush()
        tail = (tail + [line])[-60:]
        if line.startswith(SHOWN):
            print(line, end="")
        if DRIVE_RUN and line.startswith("===") and "done in" in line:
            sync(WORK, DRIVE_RUN)
RUN_OK = process.wait() == 0
if DRIVE_RUN:
    sync(WORK, DRIVE_RUN)
if not RUN_OK:
    print("".join(tail))
    print("A stage failed. The lines above show the end of its output. The next cells save what exists.")

## Results

**own** is the score of a caption on its own image. **blind %** is the share of that score that the caption keeps on a different image.
A blind share of 100% means that the metric cannot tell whether the model saw the image.
BLEU-4 and METEOR are multiplied by 100. CIDEr-D runs from 0 to 10. CLIPScore runs from 0 to 2.5.

With the `quick` profile, these scores are a wiring check. They come from very short training runs. They are not study results.

In [ ]:
# @title Scores on the 200 confirmation images
import json
import pandas as pd
from IPython.display import Image, display

LABELS = {"generic": "Image-blind control (one fixed caption)", "decoder-plain": "Our decoder, plain",
          "decoder-relation": "Our decoder, relation", "decoder-convnext": "Our decoder, ConvNeXt grid",
          "decoder-wide": "Our decoder, wide", "decoder-mobile": "Our decoder, MobileNetV3 grid",
          "decoder-dinov2": "Our decoder, DINOv2 grid",
          "blip-base": "BLIP base, zero-shot", "blip-large": "BLIP large, zero-shot",
          "blip-base-ft": "BLIP base, fine-tuned", "blip-large-ft": "BLIP large, fine-tuned"}
# A "-beam" row is the same trained weights as the row above it, decoded with three beams as BLIP is.
LABELS.update({f"{key}-beam": f"{label}, 3 beams" for key, label in list(LABELS.items()) if key.startswith("decoder-")})
summary_path = WORK / "results" / "summary.json"
if summary_path.is_file():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    if PROFILE == "quick":
        print("Quick profile: a wiring check on very short training runs. These scores are not study results.")
    names = {"bleu4_corpus": ("BLEU-4", 100, 1), "meteor": ("METEOR", 100, 1), "cider_d": ("CIDEr-D", 1, 2),
             "clipscore": ("CLIPScore", 1, 2)}
    rows = []
    for family, entry in summary["families"].items():
        row = {"condition": LABELS.get(family, family), "runs": entry["runs"]}
        for key, (label, scale, digits) in names.items():
            cell = entry["metrics"].get(key)
            if cell is None:
                continue
            row[f"{label} own"] = round(scale * cell["matched"], digits)
            share = cell.get("blind_share")
            row[f"{label} blind %"] = None if share is None else round(100 * share)
        row["tokens"] = round(entry["caption_tokens_mean"], 1)
        rows.append(row)
    display(pd.DataFrame(rows).set_index("condition"))
else:
    print("No score table exists, because the run stopped before the scoring stage.")

In [ ]:
# @title Figures
for name in ("blind-share", "finetune-split", "decoder-curves"):
    path = WORK / "report" / "figures" / f"{name}.png"
    if path.is_file():
        display(Image(filename=str(path), width=900 if name == "blind-share" else 480))
    else:
        print(f"No {name} figure exists for this run.")

In [ ]:
# @title Does the choice of the other image matter?
rules = {"cyclic shift": "grounding-gap.json", "nearest photo": "grounding-gap-nearest.json",
         "farthest photo": "grounding-gap-farthest.json"}
table = {}
for rule, filename in rules.items():
    path = WORK / "results" / filename
    if path.is_file():
        models = json.loads(path.read_text(encoding="utf-8"))["models"]
        table[rule] = {model: round(100 * entry["metrics"]["cider_d"]["blind_share"]) for model, entry in models.items()
                       if entry["metrics"].get("cider_d", {}).get("blind_share") is not None}
if len(table) > 1:
    print("CIDEr-D blind share (%) under three rules for the other image")
    display(pd.DataFrame(table))
else:
    print("Only the cyclic shift ran. The midterm and full profiles add the other two rules.")

## Real captions

The page shows 16 confirmation photos drawn with a fixed random seed. Nobody chose a photo by its result. Each caption is unedited.
Under each photo is the caption of every model that ran. The image-blind control writes the same caption for every photo, so the page leaves it out.
The photos belong to their Flickr owners. Clear this output before you publish the notebook.

In [ ]:
# @title Caption gallery
if (WORK / "gallery.png").is_file():
    display(Image(filename=str(WORK / "gallery.png"), width=1000))
else:
    print("No gallery exists, because the run stopped before that stage.")

## Download the results

The zip file holds the aggregate scores, the figures, the results table, the run log, and each model's captions.
It holds no Flickr8k photo and no reference caption, unless you include the gallery.

In [ ]:
# @title Zip and download
INCLUDE_GALLERY = False  # @param {type:"boolean"}
import zipfile
archive = REPO / "outputs" / f"captionbench-{PROFILE}-results.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zipped:
    for folder in ("results", "report", "preflight/results"):
        for path in sorted((WORK / folder).rglob("*")) if (WORK / folder).is_dir() else []:
            if path.is_file():
                zipped.write(path, path.relative_to(WORK))
    for path in sorted(WORK.glob("*.jsonl")):
        zipped.write(path, Path("predictions") / path.name)
    for path in [*sorted(WORK.glob("runs/*/*.json")), WORK / "run.log"]:
        if path.is_file():
            zipped.write(path, path.relative_to(WORK))
    if INCLUDE_GALLERY and (WORK / "gallery.pdf").is_file():
        zipped.write(WORK / "gallery.pdf", "gallery.pdf")
print(archive, f"{archive.stat().st_size / 2**20:.1f} MiB")
if DRIVE_RUN:
    shutil.copy2(archive, DRIVE_RUN.parent / archive.name)
    print("Copied to", DRIVE_RUN.parent / archive.name)
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))

## Release the GPU

This cell disconnects the runtime so that an idle GPU uses no compute units.
It waits two minutes first, so the download can finish. Turn `AUTO_DISCONNECT` off in the settings to keep the runtime.

In [ ]:
# @title Disconnect the runtime
if IN_COLAB and AUTO_DISCONNECT:
    import time
    print("The runtime disconnects in two minutes. Its files are deleted. Check that the zip file downloaded.")
    if DRIVE_RUN:
        drive.flush_and_unmount()
        print("Drive is flushed. The results are in My Drive/CaptionBench.")
    time.sleep(120)
    from google.colab import runtime
    runtime.unassign()
else:
    print("The runtime stays connected. Choose Runtime > Disconnect and delete runtime when you finish.")